# Plantilla para la Tarea online BDA02

# Nombre del alumno:

En esta tarea deberás completar las celdas que están incompletas. Se muestra el resultado esperado de la ejecución. Se trata de que implementes un proceso MapReduce que produzca ese resultado. Puedes implementar el proceso MapReduce con el lenguaje y librería que prefieras (`Bash`, Python, `mrjob` ...). Los datos de entrada del proceso son meros ejemplos y el proceso que implementes debería funcionar con esos y cualquier otro fichero de entrada que tenga la misma estructura.

## 1.- Partiendo del fichero de `notas.txt`, calcula la nota más alta obtenida por cada alumno con un proceso MapReduce.

Es decir, que si tenemos el fichero de notas:

In [32]:
%%writefile notas.txt
pedro 6 7
luis 0 4
ana 7
pedro 8 1 3
ana 5 6 7
ana 10
luis 3

Overwriting notas.txt


Se espera obtener el siguiente resultado:

![solución 1](./img/1.png)

In [2]:
%%writefile marksMR.py
#!/usr/bin/python3

from mrjob.job import MRJob
    
#Definimos una clase MrJob
class MarksMR(MRJob):
        
    # Mapper: En esta etapa aún no hay clave (_), el valor lo recibimos en la variable line
    def mapper(self, _, line):
        #Por cada línea, esta se divide en los campos que forman las columnas
        name, *marks = line.split()
        for mark in marks:            
            yield name, float(mark)
         
    #Reducer: La clave será el nombre y los valores las notas
    def reducer(self, name, marks):
        yield name, max(marks)
        
if __name__=='__main__':
    MarksMR.run()

Overwriting marksMR.py


In [4]:
! python3 marksMR.py -r hadoop notas.txt

No configs found; falling back on auto-configuration
No configs specified for hadoop runner
Looking for hadoop binary in /app/hadoop-3.3.1/bin...
Found hadoop binary: /app/hadoop-3.3.1/bin/hadoop
Using Hadoop version 3.3.1
Looking for Hadoop streaming jar in /app/hadoop-3.3.1...
Found Hadoop streaming jar: /app/hadoop-3.3.1/share/hadoop/tools/lib/hadoop-streaming-3.3.1.jar
Creating temp directory /tmp/marksMR.root.20221205.084229.686874
uploading working dir files to hdfs:///user/root/tmp/mrjob/marksMR.root.20221205.084229.686874/files/wd...
Copying other local files to hdfs:///user/root/tmp/mrjob/marksMR.root.20221205.084229.686874/files/
Running step 1 of 1...
  packageJobJar: [/tmp/hadoop-unjar5667977909965391118/] [] /tmp/streamjob2679889539920817868.jar tmpDir=null
  Connecting to ResourceManager at yarnmaster/172.18.0.4:8032
  Connecting to ResourceManager at yarnmaster/172.18.0.4:8032
  Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1670167100255_0

## 2.- Usando un proceso MapReduce muestra las 10 palabras más utilizadas en `El Quijote`.

Lo primero será descargar El Quijote:

In [5]:
! wget -O '2000-0.txt' https://www.gutenberg.org/files/2000/2000-0.txt

--2022-12-05 09:50:18--  https://www.gutenberg.org/files/2000/2000-0.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2226045 (2.1M) [text/plain]
Saving to: ‘2000-0.txt’

2000-0.txt          100%[===================>]   2.12M   120KB/s    in 15s     

2022-12-05 09:50:34 (149 KB/s) - ‘2000-0.txt’ saved [2226045/2226045]



Al igual que hicimos en la primera práctica, eliminamos aquellas líneas que son metadata y no forman parte de la obra. Sobrescribimos el fichero sin esas líneas.

In [18]:
with open('2000-0.txt') as f:
    lines = f.readlines()

head = 24
tail = 360
book = lines[head:-tail]

with open('2000-0.txt', 'w') as f:
    for line in book:
        f.write(f"{line}\n")


El resultado debería ser el mismo que el que obtuvimos en la primera práctica.

![solución 2](./img/2.png)

In [21]:
%%writefile topTenMR.py
#!/usr/bin/python3

from mrjob.job import MRJob
from mrjob.step import MRStep

import re
    
#Definimos una clase MrJob
class TopTenMR(MRJob):
    
    def mapper_split_words(self, _, line):
        words = re.split('\W+', line)
        for word in words:
            if word:
                yield word.lower(), 1
    
    def reducer_counts_words(self, word, counts):
        yield None, (sum(counts), word)
    
    
    def reducer_top_ten(self, _, words):
        words = sorted(words, key=lambda word: word[0], reverse=True)
        yield None, words[0:10]
        
    def steps(self):
        return [
            MRStep(mapper=self.mapper_split_words,
                   reducer=self.reducer_counts_words),
            MRStep(reducer=self.reducer_top_ten)
        ]
        
if __name__=='__main__':
    TopTenMR.run()

Overwriting topTenMR.py


In [22]:
! python3 topTenMR.py 2000-0.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/topTenMR.root.20221205.091213.471805
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/topTenMR.root.20221205.091213.471805/output
Streaming final output from /tmp/topTenMR.root.20221205.091213.471805/output...
null	[[20769, "que"], [18410, "de"], [18272, "y"], [10492, "la"], [9876, "a"], [8285, "en"], [8265, "el"], [6346, "no"], [4769, "los"], [4752, "se"]]
Removing temp directory /tmp/topTenMR.root.20221205.091213.471805...


## 3.- Muestra la clasificación de temporada 2021/2022 de La Liga pero únicamente de los puntos obtenidos como visitante.

En [esta Web](https://resultados.as.com/resultados/futbol/primera/2021_2022/clasificacion/) puedes consultar cuántos puntos obtuvo cada equipo fuera de casa.

Empezamos descargando el fichero de resultados de la temporada 2021/2022 y renombrándolo a `laliga2122.csv`.

In [23]:
! wget -O laliga2122.csv https://www.football-data.co.uk/mmz4281/2122/SP1.csv

--2022-12-05 10:25:13--  https://www.football-data.co.uk/mmz4281/2122/SP1.csv
Resolving www.football-data.co.uk (www.football-data.co.uk)... 217.160.0.246
Connecting to www.football-data.co.uk (www.football-data.co.uk)|217.160.0.246|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 172174 (168K) [text/csv]
Saving to: ‘laliga2122.csv’

laliga2122.csv      100%[===================>] 168.14K   625KB/s    in 0.3s    

2022-12-05 10:25:14 (625 KB/s) - ‘laliga2122.csv’ saved [172174/172174]



Se espera este resultado:

![solución 3](./img/3.png)

In [24]:
%%writefile laligaMR.py
#!/usr/bin/python3

from mrjob.job import MRJob
from mrjob.step import MRStep
    
class LaLigaMR(MRJob):
        
    # Mapper: En esta etapa aún no hay clave (_), el valor lo recibimos en la variable line
    def mapper_points(self, _, line):
        #Por cada línea, esta se divide en los campos que forman las columnas
        _, _, _, _, away_team, _, _, result, *rest = line.split(',')
        
        # Si es la cabecera no emitimos nada
        if away_team == "AwayTeam":
            return
        
        if result == 'D':
            yield away_team, 1
        elif result == 'A':
            yield away_team, 3
            
    def combiner_points(self, team, points):
        yield team, sum(points)
            
    def reducer_points(self, team, points):
        yield None, (team, sum(points))
        
    def reducer_classification(self, _, points):
        yield None, sorted(points, key=lambda t: t[1], reverse=True)
        
    def steps(self):
        return [
            MRStep(mapper=self.mapper_points,
                   combiner=self.combiner_points,
                   reducer=self.reducer_points),
            MRStep(reducer=self.reducer_classification)
        ]
         
if __name__=='__main__':
    LaLigaMR.run()

Overwriting laligaMR.py


In [25]:
! python3 laligaMR.py laliga2122.csv

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/laligaMR.root.20221205.092521.517289
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/laligaMR.root.20221205.092521.517289/output
Streaming final output from /tmp/laligaMR.root.20221205.092521.517289/output...
null	[["Real Madrid", 42], ["Barcelona", 35], ["Betis", 33], ["Ath Madrid", 30], ["Sevilla", 28], ["Sociedad", 27], ["Osasuna", 25], ["Villarreal", 23], ["Valencia", 22], ["Ath Bilbao", 21], ["Cadiz", 21], ["Celta", 21], ["Granada", 16], ["Elche", 15], ["Vallecano", 13], ["Levante", 13], ["Mallorca", 12], ["Getafe", 11], ["Espanol", 9], ["Alaves", 6]]
Removing temp directory /tmp/laligaMR.root.20221205.092521.517289...


## 4.- Muestra la diferencia de goles entre el equipo que más goles ha marcado y el que menos goles ha marcado en la temporada 2021/2022 de La Liga.

Se espera que el proceso MapReuce produzca una salida similar a la siguiente:

![solución 4](./img/4.png)

In [28]:
%%writefile laligaMR.py
#!/usr/bin/python3

from mrjob.job import MRJob
from mrjob.step import MRStep
    
class LaLigaMR(MRJob):
        
    # Mapper: En esta etapa aún no hay clave (_), el valor lo recibimos en la variable line
    def mapper_points(self, _, line):
        #Por cada línea, esta se divide en los campos que forman las columnas
        _, _, _, home_team, away_team, home_goals, away_goals,*rest = line.split(',')
        
        # Si es la cabecera no emitimos nada
        if home_team == "HomeTeam":
            return
        yield home_team, int(home_goals)
        yield away_team, int(away_goals)
            
    def combiner_points(self, team, goals):
        yield team, sum(goals)
            
    def reducer_points(self, team, goals):
        yield None, (team, sum(goals))
        
    def reducer_goals_diff(self, _, goals):
        goals_class = sorted(goals, key=lambda t: t[1], reverse=True)
        yield f'{goals_class[0][0]} vs {goals_class[-1][0]}', f'diferencia de goles {goals_class[0][1] - goals_class[-1][1]}' 
        
    def steps(self):
        return [
            MRStep(mapper=self.mapper_points,
                   combiner=self.combiner_points,
                   reducer=self.reducer_points),
            MRStep(reducer=self.reducer_goals_diff)
        ]
         
if __name__=='__main__':
    LaLigaMR.run()

Overwriting laligaMR.py


In [29]:
! python3 laligaMR.py laliga2122.csv

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/laligaMR.root.20221205.093315.269419
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/laligaMR.root.20221205.093315.269419/output
Streaming final output from /tmp/laligaMR.root.20221205.093315.269419/output...
"Real Madrid vs Alaves"	"diferencia de goles 49"
Removing temp directory /tmp/laligaMR.root.20221205.093315.269419...


## 5.- Calcula la racha de los últimos cinco partidos de cada equipo en la clasificación final de La Liga en la temporada 2021/2022.

[Observa](https://www.google.com/search?q=clasificacion+liga+2021+2022&oq=clasificacion+liga+2021+2022#sie=lg) que las últimas columnas de la clasificación muestran cuál ha sido el resultado de los últimos 5 partidos de cada equipo.

![clasificacion](./img/clasificacion.png)

Se trata de que muestres la clasificación final junto con los resultados de los últimos 5 partidos. Este ejercicio es un poco más difícil y laborioso que los otros. Si usas `mrjob` probablemente te sea útil utilizar [ordenación secundaria por valor](https://mrjob.readthedocs.io/en/latest/job.html#secondary-sort), aunque también se puede resolver sin hacer uso de ella.

Se espera este resultado:

![solución 5](./img/5.png)

In [30]:
%%writefile laligaMR.py
#!/usr/bin/python3

from mrjob.job import MRJob
from mrjob.step import MRStep
from datetime import datetime
    
class LaLigaMR(MRJob):
    
    SORT_VALUES = True
        
    # Mapper: En esta etapa aún no hay clave (_), el valor lo recibimos en la variable line
    def mapper_points(self, _, line):
        #Por cada línea, esta se divide en los campos que forman las columnas
        _, date, _, home_team, away_team, _, _, result, *rest = line.split(',')
        
        # Si es la cabecera no emitimos nada
        if home_team == "HomeTeam":
            return
        
        date = datetime.strptime(date, "%d/%m/%Y").strftime("%Y/%m/%d")

        if result == 'D':            
            yield home_team, (date, 1)
            yield away_team, (date, 1)
        elif result == 'H':
            yield home_team, (date, 3)
            yield away_team, (date, 0)
        else:
            yield home_team, (date, 0)
            yield away_team, (date, 3)
            
    def reducer_points(self, team, points):
        points = list(points)
        points = [p for date, p in points]
        five_latest_points = points[-5:]
        five_latest_points.reverse()
        yield None, (team, sum(points), five_latest_points)
    
    
    def reducer_classification(self, _, points):
            yield None, sorted(points, key=lambda t: t[1], reverse=True)
            
    def steps(self):
        return [
            MRStep(mapper=self.mapper_points, reducer=self.reducer_points),
            MRStep(reducer=self.reducer_classification)
        ]
         
if __name__=='__main__':
    LaLigaMR.run()

Overwriting laligaMR.py


In [31]:
! python3 laligaMR.py laliga2122.csv

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/laligaMR.root.20221205.094315.867039
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/laligaMR.root.20221205.094315.867039/output
Streaming final output from /tmp/laligaMR.root.20221205.094315.867039/output...
null	[["Real Madrid", 86, [1, 1, 3, 0, 3]], ["Barcelona", 73, [0, 1, 3, 3, 3]], ["Ath Madrid", 71, [3, 1, 3, 3, 0]], ["Sevilla", 70, [3, 1, 1, 1, 1]], ["Betis", 65, [1, 3, 3, 0, 1]], ["Sociedad", 62, [0, 3, 3, 0, 1]], ["Villarreal", 59, [3, 0, 3, 1, 0]], ["Ath Bilbao", 55, [0, 3, 0, 1, 3]], ["Valencia", 48, [3, 1, 0, 1, 1]], ["Osasuna", 47, [0, 0, 1, 1, 1]], ["Celta", 46, [0, 3, 0, 3, 1]], ["Elche", 42, [3, 0, 0, 0, 1]], ["Espanol", 42, [1, 1, 0, 1, 0]], ["Vallecano", 42, [0, 0, 0, 1, 1]], ["Cadiz", 39, [3, 1, 0, 3, 1]], ["Getafe", 39, [0, 1, 1, 1, 1]], ["Mallorca", 39, [3, 3, 1, 0, 0]], ["Granada", 38, [1, 0, 3, 3, 1]], ["Levante", 35, [3, 3